# Подготовка dataset

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.linear_model import LogisticRegression

## Загрузка датасета

In [ ]:
# train = pd.read_csv('/content/drive/MyDrive/ColabDatasets/train.csv', header=0)
# test = pd.read_csv('/content/drive/MyDrive/ColabDatasets/test_X.csv', header=0)

train = pd.read_csv('/content/train.csv', header=0)
test = pd.read_csv('/content/test_X.csv', header=0)

In [ ]:
TARGET = "is_fake"
ID_COL = "id"

print(train.shape, test.shape)
print("Train columns: ", train.columns)
print("Test columns: ", test.columns)

(138039, 26) (59159, 25)
Train columns:  Index(['id', 'is_fake', 'brand', 'description', 'title_name', 'category',
       'rating_1_count', 'rating_2_count', 'rating_3_count', 'rating_4_count',
       'rating_5_count', 'comments_count', 'photos_count', 'videos_count',
       'price', 'item_time_alive', 'item_count_sales7', 'item_count_sales30',
       'item_count_sales90', 'item_count_returns7', 'item_count_returns30',
       'item_count_returns90', 'item_variety_count', 'item_available_count',
       'seller_time_alive', 'seller_id'],
      dtype='object')
Test columns:  Index(['id', 'brand', 'description', 'title_name', 'category',
       'rating_1_count', 'rating_2_count', 'rating_3_count', 'rating_4_count',
       'rating_5_count', 'comments_count', 'photos_count', 'videos_count',
       'price', 'item_time_alive', 'item_count_sales7', 'item_count_sales30',
       'item_count_sales90', 'item_count_returns7', 'item_count_returns30',
       'item_count_returns90', 'item_variety_count

## Аудит данных

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138039 entries, 0 to 138038
Data columns (total 26 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    138039 non-null  int64  
 1   is_fake               138039 non-null  int64  
 2   brand                 79313 non-null   object 
 3   description           121914 non-null  object 
 4   title_name            138039 non-null  object 
 5   category              138039 non-null  object 
 6   rating_1_count        35377 non-null   float64
 7   rating_2_count        35377 non-null   float64
 8   rating_3_count        35377 non-null   float64
 9   rating_4_count        35377 non-null   float64
 10  rating_5_count        35377 non-null   float64
 11  comments_count        35377 non-null   float64
 12  photos_count          35377 non-null   float64
 13  videos_count          35377 non-null   float64
 14  price                 138039 non-null  float64
 15  

In [ ]:
print(train[TARGET].value_counts(dropna=False))

is_fake
0    127362
1     10677
Name: count, dtype: int64


In [ ]:
print(train.isna().mean().sort_values(ascending=False).head(20))

rating_3_count          0.743717
rating_4_count          0.743717
rating_2_count          0.743717
rating_1_count          0.743717
comments_count          0.743717
rating_5_count          0.743717
videos_count            0.743717
photos_count            0.743717
brand                   0.425430
description             0.116815
item_variety_count      0.005556
item_available_count    0.005556
id                      0.000000
is_fake                 0.000000
title_name              0.000000
category                0.000000
price                   0.000000
item_time_alive         0.000000
item_count_sales30      0.000000
item_count_sales7       0.000000
dtype: float64


In [ ]:
train.dtypes

,0
id,int64
is_fake,int64
brand,object
description,object
title_name,object
category,object
rating_1_count,float64
rating_2_count,float64
rating_3_count,float64
rating_4_count,float64


In [ ]:
train.isna().sum().sort_values(ascending=False)

,0
rating_3_count,102662
rating_4_count,102662
rating_2_count,102662
rating_1_count,102662
comments_count,102662
rating_5_count,102662
videos_count,102662
photos_count,102662
brand,58726
description,16125


In [ ]:
print(train["seller_id"])

0         1218
1         1374
2         1448
3          715
4          715
          ... 
138034     875
138035     875
138036     875
138037     875
138038     875
Name: seller_id, Length: 138039, dtype: int64


In [ ]:
print(train["rating_4_count"].describe())

count    35377.000000
mean         1.384741
std          5.986190
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max        319.000000
Name: rating_4_count, dtype: float64


In [ ]:
print(train["seller_time_alive"].describe())

count    138039.000000
mean        592.435239
std         468.934297
min           1.000000
25%         180.000000
50%         495.000000
75%         927.000000
max        2215.000000
Name: seller_time_alive, dtype: float64


## Выделение фичей

### brand

#### Аудит

In [ ]:
print(train["brand"].isna().mean(), test["brand"].isna().mean())
print(train["brand"].nunique(), test["brand"].nunique())

0.42543049428060187 0.3685829713145929
3661 1352


In [ ]:
train["brand"].value_counts().head(20)

,count
brand,
iQZiP,4661
ProFDetali,3841
Levsha kaluga,3009
Sony,2856
OEM,2695
OINO,2084
HUAYU,2060
BaseMarket,1240
MyPads,1143


In [ ]:
tmp = train.copy()

tmp["brand_lower"] = train["brand"].str.lower().str.strip()
tmp["brand_lower"].value_counts()

,count
brand_lower,
iqzip,4661
profdetali,3841
levsha kaluga,3009
sony,2856
oem,2695
...,...
konfulon,1
no name,1
harman,1


In [ ]:
print(tmp["brand_lower"].value_counts().describe())

count    3623.000000
mean       21.891526
std       150.665115
min         1.000000
25%         1.000000
50%         2.000000
75%         7.000000
max      4661.000000
Name: count, dtype: float64


In [ ]:
vc = tmp["brand_lower"].value_counts()
(vc < 5).mean(), (vc < 10).mean()

(np.float64(0.6720949489373448), np.float64(0.794645321556721))

In [ ]:
tmp["brand_lower"] = tmp["brand"].str.lower().str.strip()

brand_stats = (
    tmp.groupby("brand_lower")
       .agg(
           cnt=("is_fake", "size"),
           fake_rate=("is_fake", "mean")
       )
       .query("cnt >= 50")
       .sort_values("fake_rate", ascending=False)
)

brand_stats.head(15)

,cnt,fake_rate
brand_lower,,
aria,55,0.963636
philips sonicare,112,0.821429
eurocell,67,0.820896
pioneer,182,0.813187
jbl harman,75,0.720000
redmi,139,0.712230
marshall,54,0.703704
econ,80,0.687500
logitech g,417,0.647482


#### Изменения



In [ ]:
for df in (train, test):
    df["brand"] = (
        df["brand"]
        .astype(str)
        .str.lower()
        .str.strip()
        # .str.trim()
    )
    df.loc[df["brand"].isin(["nan", "none", "null", ""]), "brand"] = ""


MIN_BRAND_COUNT = 2

brand_counts = train["brand"].value_counts()
rare_brands = brand_counts[brand_counts < MIN_BRAND_COUNT].index

for df in (train, test):
    df.loc[df["brand"].isin(rare_brands), "brand"] = "__RARE_BRAND__"


print("Unique brands (train):", train["brand"].nunique())
print("Top brands:\n", train["brand"].value_counts().head())
# print("brand_freq stats:\n", train["brand_freq"].describe())


Unique brands (train): 2231
Top brands:
 brand
                 58726
iqzip             4661
profdetali        3841
levsha kaluga     3009
sony              2856
Name: count, dtype: int64


### category

#### Аудит

In [ ]:
print(train["category"].isna().mean(), test["category"].isna().mean())

0.0 0.0


In [ ]:
print(train["category"].nunique(), test["category"].nunique())

599 488


In [ ]:
print(train["category"].value_counts().tail(100))

category
Рассеиватель для фотовспышки           2
Шторка для веб-камеры                  2
Кронштейн для проектора                2
Аксессуары для соковыжималки           2
Рекламное проекционное оборудование    2
                                      ..
Планшетный компьютер Apple             1
Аксессуары для вытяжки                 1
Пылесос бытовой Dyson                  1
Пылесос вертикальный Samsung           1
Комплект Умный дом                     1
Name: count, Length: 100, dtype: int64


#### Изменения


In [ ]:
for df in (train, test):
    df["category"] = (
        df["category"]
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace("ё", "е")
    )


MIN_CATEGORY_COUNT = 5

category_counts = train["category"].value_counts()
rare_category = category_counts[category_counts < MIN_CATEGORY_COUNT].index

for df in (train, test):
    df.loc[df["category"].isin(rare_category), "category"] = "__RARE_CATEGORY__"

In [ ]:
print("Unique category (train):", train["category"].nunique())
print("Top category:\n", train["category"].value_counts())

Unique category (train): 448
Top category:
 category
дисплеи для телефонов                                            9216
корпуса для телефонов                                            8153
аккумулятор для мобильного телефона                              6875
запчасти для телевизора                                          6865
шлейфы для телефонов                                             5652
                                                                 ... 
тестер кабельный                                                    5
планшетный компьютер samsung                                        5
защитное стекло для планшетного компьютера, электронной книги       5
зарядное устройство сетевое apple                                   5
устройство аудиомонтажа                                             5
Name: count, Length: 448, dtype: int64


In [ ]:
print(train["category"].value_counts())

category
дисплеи для телефонов                                            9216
корпуса для телефонов                                            8153
аккумулятор для мобильного телефона                              6875
запчасти для телевизора                                          6865
шлейфы для телефонов                                             5652
                                                                 ... 
тестер кабельный                                                    5
планшетный компьютер samsung                                        5
защитное стекло для планшетного компьютера, электронной книги       5
зарядное устройство сетевое apple                                   5
устройство аудиомонтажа                                             5
Name: count, Length: 448, dtype: int64


### price

#### Аудит

In [ ]:
print("NaN share (train, test):")
print(train["price"].isna().mean(), test["price"].isna().mean())
print()

print("Price describe:")
print(train["price"].describe())
print()

NaN share (train, test):
0.0 0.0

Price describe:
count    138039.000000
mean        759.608679
std         161.911375
min          69.633641
25%         648.906927
50%         735.972922
75%         833.400971
max        1816.563093
Name: price, dtype: float64



In [ ]:
print("Share of price <= 0:")
print((train["price"] <= 0).mean())
print()

print("Price quantiles:")
print(train["price"].quantile([0.001, 0.01, 0.5, 0.99, 0.999]))
print()

Share of price <= 0:
0.0

Price quantiles:
0.001     272.155679
0.010     460.798598
0.500     735.972922
0.990    1244.349253
0.999    1318.544176
Name: price, dtype: float64



In [ ]:
tmp = train.copy()
tmp["price_bin"] = pd.qcut(tmp["price"], q=10, duplicates="drop")

price_stats = (
    tmp.groupby("price_bin")
       .agg(
           cnt=("is_fake", "size"),
           fake_rate=("is_fake", "mean"),
           price_min=("price", "min"),
           price_max=("price", "max"),
       )
)

print("Price bins stats:")
print(price_stats)
print()

Price bins stats:
                      cnt  fake_rate   price_min    price_max
price_bin                                                    
(69.633, 591.537]   13804   0.076717   69.633641   591.536345
(591.537, 633.583]  13804   0.024123  591.536638   633.582259
(633.583, 665.183]  13804   0.027311  633.583242   665.183301
(665.183, 702.883]  13804   0.039844  665.183329   702.882969
(702.883, 735.973]  13804   0.044987  702.883222   735.972922
(735.973, 769.268]  13803   0.073245  735.975218   769.267086
(769.268, 807.265]  13804   0.062446  769.267695   807.263677
(807.265, 859.91]   13804   0.082150  807.266352   859.909443
(859.91, 969.87]    13804   0.155897  859.910333   969.867453
(969.87, 1816.563]  13804   0.186757  969.881442  1816.563093



/tmp/ipython-input-2020458041.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby("price_bin")


In [ ]:
cat_price = (
    train.groupby("category")["price"]
         .agg(median="median", mean="mean", count="size")
         .query("count >= 50")
)

print("Cheapest categories by median price:")
print(cat_price.sort_values("median").head(10))
print()

print("Most expensive categories by median price:")
print(cat_price.sort_values("median").tail(10))
print()

Cheapest categories by median price:
                                                        median        mean  \
category                                                                     
внешний аккумулятор                                 301.073412  407.044678   
защитная пленка для смартфона                       554.071888  606.812247   
расходные материалы для ремонта мобильного теле...  561.134396  566.884859   
защитное стекло для смартфона                       572.885974  573.724637   
sim-карта                                           573.225177  598.963492   
кнопки, толкатели для телефонов                     578.601719  590.053581   
микрофоны для телефонов                             585.606599  600.096080   
антенны для телефонов                               586.100610  595.299613   
электронные компоненты-запчасти                     586.542965  609.048475   
чехол для смарт-часов, фитнес-браслета              591.549124  589.221595   

                          

In [ ]:
print("log1p(price) describe:")
print(np.log1p(train["price"]).describe())
print()

log1p(price) describe:
count    138039.000000
mean          6.612172
std           0.210650
min           4.257507
25%           6.476829
50%           6.602551
75%           6.726714
max           7.505252
Name: price, dtype: float64



#### Изменения

In [ ]:
for df in (train, test):
    df["price_log1p"] = np.log1p(df["price"])

for df in (train, test):
    df.drop(columns=["price"], inplace=True)

In [ ]:
# print(train[["price", "price_log1p", "price_rel_cat", "price_log_diff_cat"]].head())

### Рейтинги и комменты


#### Аудит

In [ ]:
rating_cols = ["rating_1_count","rating_2_count","rating_3_count","rating_4_count","rating_5_count"]

print("NaN share for rating cols:")
print(train[rating_cols].isna().mean().sort_values(ascending=False))
print()

print("NaN share for comments_count:")
print(train["comments_count"].isna().mean())
print()

print("Describe ratings:")
print(train[rating_cols].describe().T[["count","mean","std","min","50%","max"]])
print()

print("Describe comments_count:")
print(train["comments_count"].describe())
print()

NaN share for rating cols:
rating_1_count    0.743717
rating_2_count    0.743717
rating_3_count    0.743717
rating_4_count    0.743717
rating_5_count    0.743717
dtype: float64

NaN share for comments_count:
0.7437173552401858

Describe ratings:
                  count       mean        std  min  50%     max
rating_1_count  35377.0   2.152104  11.291077  0.0  0.0   670.0
rating_2_count  35377.0   0.653617   3.576020  0.0  0.0   199.0
rating_3_count  35377.0   1.230263   6.276667  0.0  0.0   329.0
rating_4_count  35377.0   1.384741   5.986190  0.0  0.0   319.0
rating_5_count  35377.0  14.878820  61.497618  0.0  2.0  4258.0

Describe comments_count:
count    35377.000000
mean        10.644628
std         48.399871
min          0.000000
25%          0.000000
50%          1.000000
75%          5.000000
max       1674.000000
Name: comments_count, dtype: float64



In [ ]:
ratings_total = train[rating_cols].sum(axis=1)

print("Share with zero total ratings:")
print((ratings_total == 0).mean())
print()

print("Share with zero comments:")
print((train["comments_count"].fillna(0) == 0).mean())
print()

Share with zero total ratings:
0.7437173552401858

Share with zero comments:
0.8284035671078464



In [ ]:
tmp = train.copy()
tmp["ratings_total"] = tmp[rating_cols].fillna(0).sum(axis=1)
tmp["comments_count0"] = tmp["comments_count"].fillna(0)

tmp["ratings_bin"] = pd.qcut(tmp["ratings_total"].rank(method="first"), q=10, duplicates="drop")
tmp["comments_bin"] = pd.qcut(tmp["comments_count0"].rank(method="first"), q=10, duplicates="drop")

print("Fake rate by ratings_total bins:")
print(tmp.groupby("ratings_bin").agg(cnt=("is_fake","size"), avg_ratings=("ratings_total","mean"), fake_rate=("is_fake","mean")))
print()

print("Fake rate by comments_count bins:")
print(tmp.groupby("comments_bin").agg(cnt=("is_fake","size"), avg_comments=("comments_count0","mean"), fake_rate=("is_fake","mean")))
print()

Fake rate by ratings_total bins:
                        cnt  avg_ratings  fake_rate
ratings_bin                                        
(0.999, 13804.8]      13804     0.000000   0.040568
(13804.8, 27608.6]    13804     0.000000   0.019849
(27608.6, 41412.4]    13804     0.000000   0.079180
(41412.4, 55216.2]    13804     0.000000   0.089467
(55216.2, 69020.0]    13804     0.000000   0.125254
(69020.0, 82823.8]    13803     0.000000   0.159531
(82823.8, 96627.6]    13804     0.000000   0.090191
(96627.6, 110431.4]   13804     0.562808   0.067299
(110431.4, 124235.2]  13804     2.820487   0.054549
(124235.2, 138039.0]  13804    48.640539   0.047595

Fake rate by comments_count bins:
                        cnt  avg_comments  fake_rate
comments_bin                                        
(0.999, 13804.8]      13804      0.000000   0.040930
(13804.8, 27608.6]    13804      0.000000   0.017676
(27608.6, 41412.4]    13804      0.000000   0.068676
(41412.4, 55216.2]    13804      0.000000  

/tmp/ipython-input-4005903956.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("ratings_bin").agg(cnt=("is_fake","size"), avg_ratings=("ratings_total","mean"), fake_rate=("is_fake","mean")))
/tmp/ipython-input-4005903956.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("comments_bin").agg(cnt=("is_fake","size"), avg_comments=("comments_count0","mean"), fake_rate=("is_fake","mean")))


In [ ]:
tmp = train.copy()
tmp[rating_cols] = tmp[rating_cols].fillna(0)

tmp["ratings_total"] = tmp[rating_cols].sum(axis=1)
tmp["rating_avg"] = (
    1*tmp["rating_1_count"] +
    2*tmp["rating_2_count"] +
    3*tmp["rating_3_count"] +
    4*tmp["rating_4_count"] +
    5*tmp["rating_5_count"]
) / (tmp["ratings_total"] + 1e-6)

tmp["bad_share"] = (tmp["rating_1_count"] + tmp["rating_2_count"]) / (tmp["ratings_total"] + 1e-6)

# Бины по avg rating
tmp["avg_bin"] = pd.qcut(tmp["rating_avg"], q=10, duplicates="drop")
print("Fake rate by rating_avg bins:")
print(tmp.groupby("avg_bin").agg(cnt=("is_fake","size"), avg_rating=("rating_avg","mean"), fake_rate=("is_fake","mean")))
print()

# Бины по доле плохих
tmp["bad_bin"] = pd.qcut(tmp["bad_share"], q=10, duplicates="drop")
print("Fake rate by bad_share bins:")
print(tmp.groupby("bad_bin").agg(cnt=("is_fake","size"), bad_share=("bad_share","mean"), fake_rate=("is_fake","mean")))
print()

Fake rate by rating_avg bins:
                  cnt  avg_rating  fake_rate
avg_bin                                     
(-0.001, 3.6]  110431    0.169908   0.084704
(3.6, 4.807]    13804    4.285715   0.034700
(4.807, 5.0]    13804    4.987280   0.061142

Fake rate by bad_share bins:
                  cnt  bad_share  fake_rate
bad_bin                                    
(-0.001, 0.1]  124322   0.001787   0.081225
(0.1, 1.0]      13717   0.429278   0.042210



/tmp/ipython-input-3644147609.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("avg_bin").agg(cnt=("is_fake","size"), avg_rating=("rating_avg","mean"), fake_rate=("is_fake","mean")))
/tmp/ipython-input-3644147609.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("bad_bin").agg(cnt=("is_fake","size"), bad_share=("bad_share","mean"), fake_rate=("is_fake","mean")))


In [ ]:
tmp = train.copy()
tmp[rating_cols] = tmp[rating_cols].fillna(0)
tmp["ratings_total"] = tmp[rating_cols].sum(axis=1)
tmp["comments_count0"] = tmp["comments_count"].fillna(0)

ratio = tmp["comments_count0"] / (tmp["ratings_total"] + 1e-6)

print("Comments per rating ratio quantiles:")
print(ratio.quantile([0, 0.5, 0.9, 0.99, 0.999]))
print()

Comments per rating ratio quantiles:
0.000    0.000000
0.500    0.000000
0.900    0.666666
0.990    1.166667
0.999    1.999999
dtype: float64



#### Изменения

In [ ]:
rating_cols = ["rating_1_count","rating_2_count","rating_3_count","rating_4_count","rating_5_count"]

for df in (train, test):
    df[rating_cols] = df[rating_cols].fillna(0)
    df["comments_count"] = df["comments_count"].fillna(0)

    r1 = df["rating_1_count"]
    r2 = df["rating_2_count"]
    r3 = df["rating_3_count"]
    r4 = df["rating_4_count"]
    r5 = df["rating_5_count"]

    total = r1 + r2 + r3 + r4 + r5
    denom = total + 1.0

    df["rating_avg"] = ((1*r1 + 2*r2 + 3*r3 + 4*r4 + 5*r5) / denom)

    df["ratings_total_log1p"] = np.log1p(total)

    df["comments_log1p"] = np.log1p(df["comments_count"])


train.drop(columns=rating_cols, inplace=True)
test.drop(columns=rating_cols, inplace=True)

train.drop(columns=["comments_count"], inplace=True)
test.drop(columns=["comments_count"], inplace=True)

### фото и видео

#### Аудит

In [ ]:
cols = ["photos_count", "videos_count"]

print("NaN share (train):")
print(train[cols].isna().mean())
print()

print("NaN share (test):")
print(test[cols].isna().mean())
print()

print("Describe (train):")
print(train[cols].describe())
print()

NaN share (train):
photos_count    0.743717
videos_count    0.743717
dtype: float64

NaN share (test):
photos_count    0.800267
videos_count    0.800267
dtype: float64

Describe (train):
       photos_count  videos_count
count  35377.000000  35377.000000
mean       4.802216      0.541369
std       17.668493      3.902814
min        0.000000      0.000000
25%        0.000000      0.000000
50%        0.000000      0.000000
75%        3.000000      0.000000
max      918.000000    241.000000



In [ ]:
print("Share <= 0 (train):")
print((train["photos_count"].fillna(0) <= 0).mean(), (train["videos_count"].fillna(0) <= 0).mean())
print()

print("Quantiles (train) photos_count:")
print(train["photos_count"].fillna(0).quantile([0, 0.5, 0.9, 0.99, 0.999]))
print()

print("Quantiles (train) videos_count:")
print(train["videos_count"].fillna(0).quantile([0, 0.5, 0.9, 0.99, 0.999]))
print()

Share <= 0 (train):
0.8798600395540391 0.9597432609624816

Quantiles (train) photos_count:
0.000      0.0
0.500      0.0
0.900      1.0
0.990     26.0
0.999    121.0
Name: photos_count, dtype: float64

Quantiles (train) videos_count:
0.000     0.0
0.500     0.0
0.900     0.0
0.990     3.0
0.999    19.0
Name: videos_count, dtype: float64



In [ ]:
tmp = train.copy()
tmp["photos0"] = tmp["photos_count"].fillna(0)

tmp["photos_bin"] = pd.qcut(tmp["photos0"].rank(method="first"), q=10, duplicates="drop")

print("Fake rate by photos_count bins:")
print(tmp.groupby("photos_bin").agg(cnt=("is_fake","size"), avg_photos=("photos0","mean"), fake_rate=("is_fake","mean")))
print()

Fake rate by photos_count bins:
                        cnt  avg_photos  fake_rate
photos_bin                                        
(0.999, 13804.8]      13804    0.000000   0.041292
(13804.8, 27608.6]    13804    0.000000   0.017314
(27608.6, 41412.4]    13804    0.000000   0.057882
(41412.4, 55216.2]    13804    0.000000   0.070052
(55216.2, 69020.0]    13804    0.000000   0.080122
(69020.0, 82823.8]    13803    0.000000   0.117583
(82823.8, 96627.6]    13804    0.000000   0.148870
(96627.6, 110431.4]   13804    0.000000   0.087656
(110431.4, 124235.2]  13804    0.201391   0.108229
(124235.2, 138039.0]  13804   12.105766   0.044480



/tmp/ipython-input-2152490254.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("photos_bin").agg(cnt=("is_fake","size"), avg_photos=("photos0","mean"), fake_rate=("is_fake","mean")))


In [ ]:
tmp = train.copy()
tmp["videos0"] = tmp["videos_count"].fillna(0)

tmp["videos_bin"] = pd.qcut(tmp["videos0"].rank(method="first"), q=10, duplicates="drop")

print("Fake rate by videos_count bins:")
print(tmp.groupby("videos_bin").agg(cnt=("is_fake","size"), avg_videos=("videos0","mean"), fake_rate=("is_fake","mean")))
print()

Fake rate by videos_count bins:
                        cnt  avg_videos  fake_rate
videos_bin                                        
(0.999, 13804.8]      13804    0.000000   0.042379
(13804.8, 27608.6]    13804    0.000000   0.016372
(27608.6, 41412.4]    13804    0.000000   0.049768
(41412.4, 55216.2]    13804    0.000000   0.059041
(55216.2, 69020.0]    13804    0.000000   0.073819
(69020.0, 82823.8]    13803    0.000000   0.110845
(82823.8, 96627.6]    13804    0.000000   0.110185
(96627.6, 110431.4]   13804    0.000000   0.119748
(110431.4, 124235.2]  13804    0.000000   0.105839
(124235.2, 138039.0]  13804    1.387424   0.085482



/tmp/ipython-input-1307410863.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("videos_bin").agg(cnt=("is_fake","size"), avg_videos=("videos0","mean"), fake_rate=("is_fake","mean")))


In [ ]:
tmp = train.copy()
tmp["photos0"] = tmp["photos_count"].fillna(0)
tmp["videos0"] = tmp["videos_count"].fillna(0)

tmp["has_media"] = ((tmp["photos0"] > 0) | (tmp["videos0"] > 0)).astype(int)
tmp["media_total"] = tmp["photos0"] + 3*tmp["videos0"]

print("has_media distribution and fake_rate:")
print(tmp.groupby("has_media").agg(cnt=("is_fake","size"), fake_rate=("is_fake","mean"), avg_photos=("photos0","mean"), avg_videos=("videos0","mean")))
print()

tmp["media_bin"] = pd.qcut(tmp["media_total"].rank(method="first"), q=10, duplicates="drop")
print("Fake rate by media_total bins:")
print(tmp.groupby("media_bin").agg(cnt=("is_fake","size"), avg_media=("media_total","mean"), fake_rate=("is_fake","mean")))
print()

has_media distribution and fake_rate:
              cnt  fake_rate  avg_photos  avg_videos
has_media                                           
0          121072   0.082273    0.000000    0.000000
1           16967   0.042200   10.012848    1.128779

Fake rate by media_total bins:
                        cnt  avg_media  fake_rate
media_bin                                        
(0.999, 13804.8]      13804   0.000000   0.041437
(13804.8, 27608.6]    13804   0.000000   0.017386
(27608.6, 41412.4]    13804   0.000000   0.058244
(41412.4, 55216.2]    13804   0.000000   0.069835
(55216.2, 69020.0]    13804   0.000000   0.080774
(69020.0, 82823.8]    13803   0.000000   0.115989
(82823.8, 96627.6]    13804   0.000000   0.150826
(96627.6, 110431.4]   13804   0.000000   0.086931
(110431.4, 124235.2]  13804   0.235004   0.108229
(124235.2, 138039.0]  13804  16.234425   0.043828



/tmp/ipython-input-3805839025.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("media_bin").agg(cnt=("is_fake","size"), avg_media=("media_total","mean"), fake_rate=("is_fake","mean")))


#### Изменения

In [ ]:
for df in (train, test):
    df["photos_count"] = df["photos_count"].fillna(0)
    df["videos_count"] = df["videos_count"].fillna(0)

    media_total = df["photos_count"] + 3 * df["videos_count"]

    df["videos_log1p"] = np.log1p(df["videos_count"])
    df["photos_log1p"] = np.log1p(df["photos_count"])

train.drop(columns=["photos_count", "videos_count"], inplace=True)
test.drop(columns=["photos_count", "videos_count"], inplace=True)

### sales / returns

#### Аудит

In [ ]:
sales_cols = [
    "item_count_sales7", "item_count_sales30", "item_count_sales90",
    "item_count_returns7", "item_count_returns30", "item_count_returns90"
]

print("NaN share:")
print(train[sales_cols].isna().mean())
print()

print("Describe sales/returns:")
print(train[sales_cols].describe().T[["count","mean","std","min","50%","max"]])
print()

NaN share:
item_count_sales7       0.0
item_count_sales30      0.0
item_count_sales90      0.0
item_count_returns7     0.0
item_count_returns30    0.0
item_count_returns90    0.0
dtype: float64

Describe sales/returns:
                         count      mean        std  min  50%      max
item_count_sales7     138039.0  0.706836  14.563641  0.0  0.0   4257.0
item_count_sales30    138039.0  2.406211  53.309926  0.0  0.0  14558.0
item_count_sales90    138039.0  5.818204  90.613313  0.0  0.0  18295.0
item_count_returns7   138039.0  0.020480   0.286811  0.0  0.0     24.0
item_count_returns30  138039.0  0.068524   0.797581  0.0  0.0     74.0
item_count_returns90  138039.0  0.167511   2.002562  0.0  0.0    300.0



In [ ]:
print("Share of zeros:")
print((train[sales_cols] == 0).mean())
print()

Share of zeros:
item_count_sales7       0.898377
item_count_sales30      0.841327
item_count_sales90      0.792464
item_count_returns7     0.987815
item_count_returns30    0.971124
item_count_returns90    0.949529
dtype: float64



In [ ]:
tmp = train.copy()

tmp["sales90"] = tmp["item_count_sales90"]
tmp["returns90"] = tmp["item_count_returns90"]

print("Share of returns > sales (should be ~0):")
print((tmp["returns90"] > tmp["sales90"]).mean())
print()

Share of returns > sales (should be ~0):
0.0003911937930584835



In [ ]:
tmp = train.copy()
tmp["sales90"] = tmp["item_count_sales90"]
tmp["returns90"] = tmp["item_count_returns90"]

tmp["sales90_bin"] = pd.qcut(tmp["sales90"].rank(method="first"), q=10, duplicates="drop")
tmp["returns90_bin"] = pd.qcut(tmp["returns90"].rank(method="first"), q=10, duplicates="drop")

print("Fake rate by sales90 bins:")
print(tmp.groupby("sales90_bin").agg(cnt=("is_fake","size"), avg_sales=("sales90","mean"), fake_rate=("is_fake","mean")))
print()

print("Fake rate by returns90 bins:")
print(tmp.groupby("returns90_bin").agg(cnt=("is_fake","size"), avg_returns=("returns90","mean"), fake_rate=("is_fake","mean")))
print()

Fake rate by sales90 bins:


/tmp/ipython-input-4248847337.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("sales90_bin").agg(cnt=("is_fake","size"), avg_sales=("sales90","mean"), fake_rate=("is_fake","mean")))
/tmp/ipython-input-4248847337.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("returns90_bin").agg(cnt=("is_fake","size"), avg_returns=("returns90","mean"), fake_rate=("is_fake","mean")))


                        cnt  avg_sales  fake_rate
sales90_bin                                      
(0.999, 13804.8]      13804   0.000000   0.037888
(13804.8, 27608.6]    13804   0.000000   0.020139
(27608.6, 41412.4]    13804   0.000000   0.065923
(41412.4, 55216.2]    13804   0.000000   0.079252
(55216.2, 69020.0]    13804   0.000000   0.113808
(69020.0, 82823.8]    13803   0.000000   0.136782
(82823.8, 96627.6]    13804   0.000000   0.099899
(96627.6, 110431.4]   13804   0.075340   0.114097
(110431.4, 124235.2]  13804   1.900391   0.051289
(124235.2, 138039.0]  13804  56.205882   0.054405

Fake rate by returns90 bins:
                        cnt  avg_returns  fake_rate
returns90_bin                                      
(0.999, 13804.8]      13804     0.000000   0.041799
(13804.8, 27608.6]    13804     0.000000   0.015793
(27608.6, 41412.4]    13804     0.000000   0.051000
(41412.4, 55216.2]    13804     0.000000   0.059186
(55216.2, 69020.0]    13804     0.000000   0.077514
(69020

In [ ]:
tmp = train.copy()
tmp["sales90"] = tmp["item_count_sales90"]
tmp["returns90"] = tmp["item_count_returns90"]

tmp["return_rate_90"] = tmp["returns90"] / (tmp["sales90"] + 1)

tmp["rr_bin"] = pd.qcut(tmp["return_rate_90"].rank(method="first"), q=10, duplicates="drop")

print("Fake rate by return_rate_90 bins:")
print(tmp.groupby("rr_bin").agg(cnt=("is_fake","size"), avg_rr=("return_rate_90","mean"), fake_rate=("is_fake","mean")))
print()

Fake rate by return_rate_90 bins:
                        cnt   avg_rr  fake_rate
rr_bin                                         
(0.999, 13804.8]      13804  0.00000   0.041799
(13804.8, 27608.6]    13804  0.00000   0.015793
(27608.6, 41412.4]    13804  0.00000   0.051000
(41412.4, 55216.2]    13804  0.00000   0.059186
(55216.2, 69020.0]    13804  0.00000   0.077514
(69020.0, 82823.8]    13803  0.00000   0.107296
(82823.8, 96627.6]    13804  0.00000   0.108954
(96627.6, 110431.4]   13804  0.00000   0.126775
(110431.4, 124235.2]  13804  0.00000   0.101710
(124235.2, 138039.0]  13804  0.07124   0.083454



/tmp/ipython-input-985389770.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby("rr_bin").agg(cnt=("is_fake","size"), avg_rr=("return_rate_90","mean"), fake_rate=("is_fake","mean")))


#### Изменения

In [ ]:
for df in (train, test):

    s7  = df["item_count_sales7"].fillna(0)
    s30 = df["item_count_sales30"].fillna(0)
    s90 = df["item_count_sales90"].fillna(0)

    r7  = df["item_count_returns7"].fillna(0)
    r30 = df["item_count_returns30"].fillna(0)
    r90 = df["item_count_returns90"].fillna(0)

    df["activity_level"] = np.log1p(s90)
    df["activity_concentration"] = s7 / (s90 + 1)
    df["sales_trend"] = np.log1p(s7 / 7) - np.log1p(s30 / 30)
    df["return_rate"] = r90 / (s90 + 1)
    df["return_spike"] = (r7 / (s7 + 1)) - df["return_rate"]

drop_cols = [
    "item_count_sales7", "item_count_sales30", "item_count_sales90",
    "item_count_returns7", "item_count_returns30", "item_count_returns90",
]

train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

### Жизненный цикл

#### Аудит


In [ ]:
print("NaN share:")
print(train["item_time_alive"].isna().mean())
print()

print("Describe:")
print(train["item_time_alive"].describe())
print()

NaN share:
0.0

Describe:
count    138039.000000
mean        209.536812
std         293.766445
min           0.000000
25%           4.000000
50%          59.000000
75%         328.000000
max        2137.000000
Name: item_time_alive, dtype: float64



In [ ]:
print("Quantiles:")
print(train["item_time_alive"].quantile([0, 0.01, 0.1, 0.5, 0.9, 0.99, 0.999]))
print()

Quantiles:
0.000       0.000
0.010       1.000
0.100       1.000
0.500      59.000
0.900     661.000
0.990    1181.000
0.999    1715.658
Name: item_time_alive, dtype: float64



In [ ]:
tmp = train.copy()
tmp["alive_bin"] = pd.qcut(
    tmp["item_time_alive"].rank(method="first"),
    q=10,
    duplicates="drop"
)

print("Fake rate by item_time_alive bins:")
print(
    tmp.groupby("alive_bin")
       .agg(
           cnt=("is_fake","size"),
           avg_days=("item_time_alive","mean"),
           fake_rate=("is_fake","mean")
       )
)
print()

Fake rate by item_time_alive bins:
                        cnt    avg_days  fake_rate
alive_bin                                         
(0.999, 13804.8]      13804    0.921110   0.153724
(13804.8, 27608.6]    13804    2.021298   0.108157
(27608.6, 41412.4]    13804    4.359461   0.106418
(41412.4, 55216.2]    13804   10.354173   0.103521
(55216.2, 69020.0]    13804   33.998624   0.120255
(69020.0, 82823.8]    13803   94.646454   0.054916
(82823.8, 96627.6]    13804  190.331643   0.046508
(96627.6, 110431.4]   13804  329.032092   0.038540
(110431.4, 124235.2]  13804  529.913866   0.024631
(124235.2, 138039.0]  13804  899.781078   0.016807



/tmp/ipython-input-2843641215.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby("alive_bin")


In [ ]:
print("NaN share:")
print(train["item_variety_count"].isna().mean())
print()

print("Describe:")
print(train["item_variety_count"].describe())
print()

NaN share:
0.005556400727330682

Describe:
count    137272.000000
mean         60.992628
std         198.320793
min           1.000000
25%           1.000000
50%           4.000000
75%          18.000000
max        1410.000000
Name: item_variety_count, dtype: float64



In [ ]:
print("Share == 0:", (train["item_variety_count"] == 0).mean())
print("Share == 1:", (train["item_variety_count"] == 1).mean())
print()

Share == 0: 0.0
Share == 1: 0.280514926940937



In [ ]:
tmp = train.copy()
tmp["variety_bin"] = pd.qcut(
    tmp["item_variety_count"].rank(method="first"),
    q=10,
    duplicates="drop"
)

print("Fake rate by item_variety_count bins:")
print(
    tmp.groupby("variety_bin")
       .agg(
           cnt=("is_fake","size"),
           avg_variety=("item_variety_count","mean"),
           fake_rate=("is_fake","mean")
       )
)
print()

Fake rate by item_variety_count bins:
                        cnt  avg_variety  fake_rate
variety_bin                                        
(0.999, 13728.1]      13728     1.000000   0.055070
(13728.1, 27455.2]    13727     1.000000   0.112770
(27455.2, 41182.3]    13727     1.179209   0.130837
(41182.3, 54909.4]    13727     2.000000   0.087929
(54909.4, 68636.5]    13727     3.247250   0.059591
(68636.5, 82363.6]    13727     5.126539   0.069935
(82363.6, 96090.7]    13727     9.147155   0.065564
(96090.7, 109817.8]   13727    18.787499   0.053836
(109817.8, 123544.9]  13727    52.055220   0.052670
(123544.9, 137272.0]  13728   516.354604   0.071678



/tmp/ipython-input-1448854783.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby("variety_bin")


In [ ]:
print("NaN share:")
print(train["item_available_count"].isna().mean())
print()

print("Describe:")
print(train["item_available_count"].describe())
print()

NaN share:
0.005556400727330682

Describe:
count    137272.000000
mean         60.975108
std         198.325075
min           0.000000
25%           1.000000
50%           4.000000
75%          18.000000
max        1410.000000
Name: item_available_count, dtype: float64



In [ ]:
print("Share == 0:", (train["item_available_count"] == 0).mean())
print()

print("Quantiles:")
print(train["item_available_count"].quantile([0, 0.5, 0.9, 0.99, 0.999]))
print()

Share == 0: 0.0017386390802599265

Quantiles:
0.000       0.0
0.500       4.0
0.900     102.0
0.990    1410.0
0.999    1410.0
Name: item_available_count, dtype: float64



In [ ]:
tmp = train.copy()
tmp["avail_bin"] = pd.qcut(
    tmp["item_available_count"].rank(method="first"),
    q=10,
    duplicates="drop"
)

print("Fake rate by item_available_count bins:")
print(
    tmp.groupby("avail_bin")
       .agg(
           cnt=("is_fake","size"),
           avg_available=("item_available_count","mean"),
           fake_rate=("is_fake","mean")
       )
)
print()

Fake rate by item_available_count bins:
                        cnt  avg_available  fake_rate
avail_bin                                            
(0.999, 13728.1]      13728       0.982517   0.054706
(13728.1, 27455.2]    13727       1.000000   0.111168
(27455.2, 41182.3]    13727       1.165003   0.133168
(41182.3, 54909.4]    13727       2.000000   0.086399
(54909.4, 68636.5]    13727       3.225905   0.060756
(68636.5, 82363.6]    13727       5.104174   0.070008
(82363.6, 96090.7]    13727       9.115612   0.065491
(96090.7, 109817.8]   13727      18.734174   0.053836
(109817.8, 123544.9]  13727      52.040286   0.052670
(123544.9, 137272.0]  13728     516.354604   0.071678



/tmp/ipython-input-4233894643.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby("avail_bin")


#### Изменения

In [ ]:
for df in (train, test):
    df["item_variety_count"] = df["item_variety_count"].fillna(0)
    df["item_available_count"] = df["item_available_count"].fillna(0)
    df["item_time_alive"] = df["item_time_alive"].fillna(0)

    df["item_variety_log1p"] = np.log1p(df["item_variety_count"])
    df["item_available_log1p"] = np.log1p(df["item_available_count"])
    df["item_time_alive_log1p"] = np.log1p(df["item_time_alive"])


for df in (train, test):
    df.drop(columns=["item_available_count"], inplace=True)
    df.drop(columns=["item_time_alive"], inplace=True)
    df.drop(columns=["item_variety_count"], inplace=True)

### seller

#### Аудит

In [ ]:
print("NaN share:")
print(train["seller_time_alive"].isna().mean())
print()

print("Describe:")
print(train["seller_time_alive"].describe())
print()

NaN share:
0.0

Describe:
count    138039.000000
mean        592.435239
std         468.934297
min           1.000000
25%         180.000000
50%         495.000000
75%         927.000000
max        2215.000000
Name: seller_time_alive, dtype: float64



In [ ]:
print("Quantiles:")
print(train["seller_time_alive"].quantile([0, 0.01, 0.1, 0.5, 0.9, 0.99, 0.999]))
print()

Quantiles:
0.000       1.000
0.010       5.000
0.100      47.000
0.500     495.000
0.900    1235.000
0.990    1803.000
0.999    2118.962
Name: seller_time_alive, dtype: float64



In [ ]:
tmp = train.copy()
tmp["seller_alive_bin"] = pd.qcut(
    tmp["seller_time_alive"].rank(method="first"),
    q=10,
    duplicates="drop"
)

print("Fake rate by seller_time_alive bins:")
print(
    tmp.groupby("seller_alive_bin")
       .agg(
           cnt=("is_fake","size"),
           avg_days=("seller_time_alive","mean"),
           fake_rate=("is_fake","mean")
       )
)
print()

Fake rate by seller_time_alive bins:
                        cnt     avg_days  fake_rate
seller_alive_bin                                   
(0.999, 13804.8]      13804    18.138511   0.155317
(13804.8, 27608.6]    13804    95.670168   0.116923
(27608.6, 41412.4]    13804   181.747827   0.073602
(41412.4, 55216.2]    13804   306.011881   0.083744
(55216.2, 69020.0]    13804   422.178427   0.062735
(69020.0, 82823.8]    13803   569.202565   0.058248
(82823.8, 96627.6]    13804   743.781947   0.091930
(96627.6, 110431.4]   13804   937.493263   0.059548
(110431.4, 124235.2]  13804  1148.496523   0.034917
(124235.2, 138039.0]  13804  1501.629600   0.036511



/tmp/ipython-input-1328100636.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby("seller_alive_bin")


In [ ]:
print("NaN share:")
print(train["seller_id"].isna().mean(), test["seller_id"].isna().mean())
print()

print("Unique sellers (train, test):")
print(train["seller_id"].nunique(), test["seller_id"].nunique())
print()

NaN share:
0.0 0.0

Unique sellers (train, test):
9758 3629



In [ ]:
vc = train["seller_id"].value_counts()

print("Seller frequency describe:")
print(vc.describe())
print()

print("Top sellers:")
print(vc.head(10))
print()

Seller frequency describe:
count    9758.000000
mean       14.146239
std        87.403350
min         1.000000
25%         1.000000
50%         2.000000
75%         6.000000
max      5202.000000
Name: count, dtype: float64

Top sellers:
seller_id
24     5202
442    3009
6      2649
43     1464
49     1316
296    1247
28     1234
15     1176
212    1106
66     1055
Name: count, dtype: int64



In [ ]:
seller_stats = (
    train.groupby("seller_id")
         .agg(
             cnt=("is_fake","size"),
             fake_rate=("is_fake","mean")
         )
         .query("cnt >= 50")
         .sort_values("fake_rate", ascending=False)
)

print("Top sellers by fake_rate:")
print(seller_stats.head(15))
print()

Top sellers by fake_rate:
           cnt  fake_rate
seller_id                
47          61   1.000000
75         109   1.000000
1256       397   1.000000
501        141   1.000000
2086       104   0.990385
5268       119   0.974790
2974        55   0.963636
2872       116   0.939655
793        104   0.932692
9          236   0.923729
1104        68   0.823529
255        148   0.817568
799        135   0.785185
515        202   0.727723
107         91   0.725275



In [ ]:
print("Share of sellers with fake_rate > 0.5 (cnt>=50):")
print((seller_stats["fake_rate"] > 0.5).mean())
print()

Share of sellers with fake_rate > 0.5 (cnt>=50):
0.05106382978723404



#### Изменения

In [ ]:
for df in (train, test):
    df["seller_time_alive_log1p"] = np.log1p(df["seller_time_alive"] + 1)

train.drop(columns=["seller_time_alive"], inplace=True)
test.drop(columns=["seller_time_alive"], inplace=True)

for df in (train, test):
    df["seller_id"] = df["seller_id"].astype(str)

# min_seller_count = 10
# seller_counts = train["seller_id"].value_counts()
# rare_sellers = seller_counts[seller_counts < min_seller_count].index

# for df in (train, test):
#     df.loc[df["seller_id"].isin(rare_sellers), "seller_id"] = "__RARE_SELLER__"

### Description


#### Аудит

In [ ]:
txt = train.loc[0, "description"]
print("len:", len(txt))
print("last 80 chars:", repr(txt[-80:]))

len: 886
last 80 chars: ' TRIATHLON, FC6841 - FC6845 TRIATHLONОдноразовые мешки-пылесборники ACTRUM изгот'


In [ ]:
lens = train["description"].fillna("").str.len()
print(lens.describe())
print(lens.quantile([0.9, 0.95, 0.99, 0.999]))


count    138039.000000
mean        370.255203
std         267.292522
min           0.000000
25%          74.000000
50%         477.000000
75%         584.000000
max        1024.000000
Name: description, dtype: float64
0.900     632.0
0.950     710.0
0.990     956.0
0.999    1007.0
Name: description, dtype: float64


#### Изменения

In [ ]:
import re
import html

def clean_text(text: str) -> str:
    if not text:
        return ""

    # html entities -> normal text (&nbsp; -> space)
    text = html.unescape(text)

    # remove html tags
    text = re.sub(r"<[^>]+>", " ", text)

    # lowercase
    text = text.lower()

    # replace non-letter/digit with space
    text = re.sub(r"[^a-zа-я0-9]+", " ", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

def fix_glued(text: str) -> str:
    if not text:
        return ""
    text = text.lower()
    # пробел между латиницей/кириллицей
    text = re.sub(r"([a-z])([а-я])", r"\1 \2", text)
    text = re.sub(r"([а-я])([a-z])", r"\1 \2", text)
    # пробел между буквами и цифрами
    text = re.sub(r"([a-zа-я])(\d)", r"\1 \2", text)
    text = re.sub(r"(\d)([a-zа-я])", r"\1 \2", text)
    # нормализация пробелов
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
for df in (train, test):
    df["description"]  = df["description"].fillna("")

    df["description"] = df["description"].apply(fix_glued)
    df["description"] = df["description"].apply(clean_text)

    df["desc_len_log1p"] = np.log1p(df["description"].str.len())

### Title name


#### Аудит

In [ ]:
train["title_name"].isna().mean()

np.float64(0.0)

In [ ]:
train["title_name"].str.len().describe(percentiles=[.01, .05, .25, .5, .75, .95, .99])

,title_name
count,138039.000000
mean,67.251646
std,34.990526
min,2.000000
1%,17.000000
5%,28.000000
25%,45.000000
50%,59.000000
75%,79.000000
95%,146.000000


In [ ]:
train["title_name"].fillna("").str.strip().eq("").mean()

np.float64(0.0)

In [ ]:
train["title_name"].str.lower().value_counts().describe()

,count
count,111314.000000
mean,1.240087
std,2.517440
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,395.000000


In [ ]:
dup_share = train["title_name"].duplicated().mean()
dup_share

np.float64(0.18901180101275727)

In [ ]:
train.groupby("title_name")["is_fake"].agg(["count", "mean"]) \
     .sort_values("count", ascending=False).tail(1000)

,count,mean
title_name,,
Дисплей Redmi 9A / Redmi 9C / Redmi 10A ориг. без рамы,1,1.0
Дисплей Premium LCD MyPads для Xiaomi Redmi 5 Plus / 1540358813,1,0.0
Дисплей SAMSUNG T100 /модуль 2 дисплея/ (арт.56565),1,0.0
Дисплей Samsung Galaxy A05 (A055F) в сборе с тачскрином Оригинал,1,0.0
Дисплей Samsung Galaxy A04 (A045F) в сборе с тачскрином черный,1,0.0
...,...,...
"Дисплей (Экран) для iPhone 8,белый, с тачскрином (Incell)",1,0.0
"Дисплей (Экран) для iPhone 7,белый, с тачскрином (Incell)",1,0.0
"Дисплей (Экран) для iPhone 7 Plus , белый , с тачскрином (Incell)",1,0.0


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(
    ngram_range=(1,2),
    min_df=20,
    stop_words=None
)

X = cv.fit_transform(train["title_name"].fillna(""))
freq = X.sum(axis=0).A1
tokens = cv.get_feature_names_out()

pd.DataFrame({"token": tokens, "freq": freq}) \
  .sort_values("freq", ascending=False).head(50)


,token,freq
7898,для,88217
5811,samsung,32475
7596,гб,23343
11412,черный,17161
6890,xiaomi,16118
7941,для samsung,15942
5291,pro,13924
5079,philips,11653
7953,для xiaomi,10732
3254,galaxy,10584


In [ ]:
common_titles = set(train["title_name"]) & set(test["title_name"])
len(common_titles) / train["title_name"].nunique()

0.06160002858470004

#### Изменения

In [ ]:
for df in (train, test):
    df["title_len_log1p"] = np.log1p(df["title_name"].str.len()).astype(np.float64)

    # df["title_name"] = df["title_name"].apply(fix_glued)
    # df["title_name"] = df["title_name"].apply(clean_text)

## Финальная проверка

In [ ]:
print("Train shape:", train.shape)
print("Test shape :", test.shape)
print()

print("Train columns:")
print(train.columns.tolist())
print()

Train shape: (138039, 24)
Test shape : (59159, 23)

Train columns:
['id', 'is_fake', 'brand', 'description', 'title_name', 'category', 'seller_id', 'price_log1p', 'rating_avg', 'ratings_total_log1p', 'comments_log1p', 'videos_log1p', 'photos_log1p', 'activity_level', 'activity_concentration', 'sales_trend', 'return_rate', 'return_spike', 'item_variety_log1p', 'item_available_log1p', 'item_time_alive_log1p', 'seller_time_alive_log1p', 'desc_len_log1p', 'title_len_log1p']



In [ ]:
print("Target distribution:")
print(train["is_fake"].value_counts(normalize=True))
print()

Target distribution:
is_fake
0    0.922652
1    0.077348
Name: proportion, dtype: float64



## Разделение на признаки и таргет

In [ ]:
text_cols = ["title_name", "description", ]
cat_cols  = ["seller_id", "category", "brand"]
num_cols  = [c for c in train.columns if c not in [ID_COL, TARGET] + text_cols + cat_cols]

In [ ]:
print("Categorical features:")
print(cat_cols)
print()

print("Numerical features:")
print(num_cols)
print()

Categorical features:
['seller_id', 'category', 'brand']

Numerical features:
['price_log1p', 'rating_avg', 'ratings_total_log1p', 'comments_log1p', 'videos_log1p', 'photos_log1p', 'activity_level', 'activity_concentration', 'sales_trend', 'return_rate', 'return_spike', 'item_variety_log1p', 'item_available_log1p', 'item_time_alive_log1p', 'seller_time_alive_log1p', 'desc_len_log1p', 'title_len_log1p']



# Обучение

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138039 entries, 0 to 138038
Data columns (total 24 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       138039 non-null  int64  
 1   is_fake                  138039 non-null  int64  
 2   brand                    138039 non-null  object 
 3   description              138039 non-null  object 
 4   title_name               138039 non-null  object 
 5   category                 138039 non-null  object 
 6   seller_id                138039 non-null  object 
 7   price_log1p              138039 non-null  float64
 8   rating_avg               138039 non-null  float64
 9   ratings_total_log1p      138039 non-null  float64
 10  comments_log1p           138039 non-null  float64
 11  videos_log1p             138039 non-null  float64
 12  photos_log1p             138039 non-null  float64
 13  activity_level           138039 non-null  float64
 14  acti

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import SGDClassifier

to_1d = FunctionTransformer(lambda x: x.ravel(), validate=False)

text_title = Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="")),
    ("to1d", to_1d),
    ("union", FeatureUnion([
        ("char", TfidfVectorizer(
            analyzer="char",
            ngram_range=(3,5),
            min_df=10,
            max_features=200_000,
            sublinear_tf=True,
            lowercase=True
        )),
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1,2),
            min_df=3,
            max_features=100_000,
            sublinear_tf=True,
            lowercase=True
        )),
    ])),
])

text_desc = Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="")),
    ("to1d", to_1d),
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(4,6),
        min_df=15,
        max_features=250_000,
        sublinear_tf=True,
        lowercase=True,
        max_df=0.95
    )),
])

text_category = Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="")),
    ("to1d", to_1d),
    ("tfidf", TfidfVectorizer(
        ngram_range=(2,3),
        min_df=10,
        max_features=150_000,
        sublinear_tf=True,
        lowercase=True
    )),
])

text_brand = Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="")),
    ("to1d", to_1d),
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,1),
        min_df=5,
        max_features=150_000,
        sublinear_tf=True,
        lowercase=True
    )),
])

cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True,
        # min_frequency=5,
    )),
])

num_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("title", text_title, ["title_name"]),
        ("desc",  text_desc,  ["description"]),
        ("category_",  text_category,  ["category"]),
        ("brand_",  text_brand,  ["brand"]),
        ("cat",   cat_pipe,   cat_cols),
        ("num",   num_pipe,   num_cols),
    ],
    remainder="drop",
)

sgd = SGDClassifier(
    loss="log_loss",
    penalty="l2", # elasticnet + l1_ratio 0.05, 0.1, 0.2
    alpha=3e-6,            # 1e-6, 3e-6, 1e-5, 3e-5, 1e-4
    max_iter=10000,
    tol=1e-3, #1e-4
    class_weight="balanced",
    random_state=42,
)

clf = Pipeline([
    ("prep", preprocess),
    ("clf", sgd),
])

In [ ]:
X = train.drop(columns=[TARGET, ID_COL]).copy()
y = train[TARGET].to_numpy()

In [ ]:
# scores, oof = run_cv(
#     model=clf,
#     df=train,
#     target_col="is_fake",
#     n_splits=2,
#     group_col="seller_id",
#     use_groups=True,
# )

In [ ]:
# Fold 1: AUC=0.96355  PR-AUC=0.79572  train_size=117592 val_size=20447  train_groups=1510 val_groups=285

In [ ]:
# fold 1: AP = 0.86021
# fold 2: AP = 0.85836
# fold 3: AP = 0.85343
# CV mean AP = 0.85733 ± 0.00286
# OOF AP     = 0.73762

# Predict

In [ ]:
clf.fit(X, y)
test_pred = clf.predict_proba(test)[:, 1]

# Сохранние submission

In [ ]:
submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET: test_pred
})

submission.to_csv("submission.csv", index=False)
print(submission.head())

       id       is_fake
0  138039  3.169794e-12
1  138040  1.428548e-15
2  138041  8.398768e-09
3  138042  3.325010e-08
4  138043  2.303143e-11
